# Pinwheel of blobs: separability of a three-class dataset

Each class is a point cloud filling a lumpy blob stretched along the stick from $(a,1)$ to $(b,1)$; the three blobs are rotations by $0^\circ$, $120^\circ$, $240^\circ$.
Sliders: `a`, `b` (the stick), `width` (half-thickness), `wobble` (lumpiness). The dashed outline is each class's convex hull.

The title reports
* whether the three hulls are pairwise disjoint,
* which classes split off from the other two by a hyperplane,
* whether the dataset is separable: a centered $W\ne0$ with $y_i\ge y_j$ for every $x$ in class $i$ and $j\ne i$.

When separable, the regions $R_i=\{y_i=\max y\}$ of a separating $W$ are shaded.
Separability is decided by the LP: centered $W$, all margins $(y_i-y_j)(x)\ge0$, margins summing to $1$ (valid since the dataset is full rank).

Requires `numpy`, `scipy`, `matplotlib`, `ipywidgets`.

In [3]:
%matplotlib inline
import numpy as np
from scipy.optimize import linprog

RNG = np.random.default_rng(0)
NB, NI = 48, 40                                   # boundary / interior points per blob
_T = np.linspace(0, 2 * np.pi, NB, endpoint=False)
_U = RNG.uniform(size=(NI, 2))                    # fixed so the picture doesn't jitter


def rot(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s], [s, c]])


def blobs(a, b, width, wobble):
    """Three classes: elliptical blobs along the stick from (a,1) to (b,1), rotated by 0, 120, 240 deg.
    Semi-major axis (b-a)/2, semi-minor `width`; `wobble` bends the boundary into a lumpier blob."""
    cx, L = (a + b) / 2, (b - a) / 2
    r = 1 + wobble * (0.5 * np.sin(3 * _T) + 0.3 * np.cos(5 * _T + 1))
    bd = np.stack([cx + L * r * np.cos(_T), 1 + width * r * np.sin(_T)], 1)
    rr, th = np.sqrt(_U[:, 0]) * 0.9, 2 * np.pi * _U[:, 1]
    rin = 1 + wobble * (0.5 * np.sin(3 * th) + 0.3 * np.cos(5 * th + 1))
    inn = np.stack([cx + L * rr * rin * np.cos(th), 1 + width * rr * rin * np.sin(th)], 1)
    base = np.vstack([bd, inn])
    return [base @ rot(2 * np.pi * i / 3).T for i in range(3)]


def lift(P):
    return np.hstack([P, np.ones((len(P), 1))])


def hulls_disjoint(A, B):
    """Compact hulls are disjoint iff strictly separable: w.x+b >= 1 on A, <= -1 on B."""
    Z = np.vstack([lift(A), -lift(B)])
    res = linprog(np.zeros(3), A_ub=-Z, b_ub=-np.ones(len(Z)), bounds=[(None, None)] * 3, method="highs")
    return res.status == 0


def pairwise_disjoint(C):
    return all(hulls_disjoint(C[i], C[j]) for i in range(3) for j in range(i + 1, 3))


def split_weight(C, m):
    """(w,b) != 0 with class m in {>= 0} and the rest in {<= 0} (margins normalized to sum 1), or None."""
    Z = np.vstack([lift(C[m])] + [-lift(C[i]) for i in range(3) if i != m])
    res = linprog(np.zeros(3), A_ub=-Z, b_ub=np.zeros(len(Z)), A_eq=Z.sum(0, keepdims=True), b_eq=[1.0],
                  bounds=[(None, None)] * 3, method="highs")
    return res.x if res.status == 0 else None


def splits_off(C, m):
    return split_weight(C, m) is not None


def one_vs_rest_weight(C, m):
    """The degenerate centered separating W splitting off class m: rows 2u/3 for m, -u/3 for the others."""
    u = split_weight(C, m)
    if u is None:
        return None
    W = np.tile(-u / 3, (3, 1))
    W[m] = 2 * u / 3
    return W


def shape_of(W, tol=1e-7):
    """Which picture W gives: 'one vs rest' (some h_ij = 0), 'slab' (parallel w_i - w_j), or 'wedges'."""
    s = np.abs(W).max()
    D = [W[i] - W[j] for i, j in [(0, 1), (1, 2), (0, 2)]]
    if any(np.abs(d).max() < tol * s for d in D):
        return "one vs rest"
    cr = D[0][0] * D[1][1] - D[0][1] * D[1][0]
    return "slab" if abs(cr) < tol * s ** 2 else "wedges"


def separating_weight(C):
    """A centered separating W (rows (w_i, b_i)), or None. LP: all margins >= 0, sum of margins = 1."""
    rows = []
    for i in range(3):
        Z = lift(C[i])
        for j in range(3):
            if j != i:
                G = np.zeros((len(Z), 9))
                G[:, 3 * i:3 * i + 3] += Z
                G[:, 3 * j:3 * j + 3] -= Z
                rows.append(G)
    G = np.vstack(rows)
    A_eq = np.vstack([G.sum(0)] + [np.tile(np.eye(3)[k], 3) for k in range(3)])
    res = linprog(np.zeros(9), A_ub=-G, b_ub=np.zeros(len(G)), A_eq=A_eq, b_eq=[1.0, 0, 0, 0],
                  bounds=[(None, None)] * 9, method="highs")
    return res.x.reshape(3, 3) if res.status == 0 else None

In [4]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.spatial import ConvexHull
from ipywidgets import interact, FloatSlider, Dropdown

COLORS = ["red", "green", "blue"]
LIM = 4.0
g = np.linspace(-LIM, LIM, 400)
X1, X2 = np.meshgrid(g, g)
GRID = np.stack([X1.ravel(), X2.ravel(), np.ones(X1.size)], axis=1)
CMAP = ListedColormap(["#ffd6d6", "#d6f5d6", "#d6dcff", "#e4e4e4"])  # last: merged region R_j = R_k


def show(a=-2.5, b=1.3, width=0.3, wobble=0.15, draw="prefer one vs rest"):
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal")
    if b <= a:
        ax.set_title("need a < b"); plt.show(); return
    C = blobs(a, b, width, wobble)
    disjoint = pairwise_disjoint(C)
    ovr = [m for m in range(3) if splits_off(C, m)]
    W = separating_weight(C)
    if W is not None and ovr and draw == "prefer one vs rest":
        W = one_vs_rest_weight(C, ovr[0])
    if W is not None:
        Y = GRID @ W.T
        R = np.argmax(Y, axis=1)
        if shape_of(W) == "one vs rest":          # two rows coincide: R_j = R_k, shade it grey
            m = int(np.argmax([np.abs(W[i] - W).sum() for i in range(3)]))
            R = np.where(R == m, m, 3)
        R = R.reshape(X1.shape)
        ax.imshow(R, extent=(-LIM, LIM, -LIM, LIM), origin="lower", cmap=CMAP,
                  vmin=0, vmax=3, alpha=0.9, interpolation="nearest")
        verdict = f"SEPARABLE; shaded: regions $R_i$ of a separating $W$ ({shape_of(W)})"
    else:
        verdict = "NOT SEPARABLE"
    for P, c in zip(C, COLORS):
        h = P[ConvexHull(P).vertices]
        ax.fill(h[:, 0], h[:, 1], color=c, alpha=0.12)
        ax.plot(*np.vstack([h, h[:1]]).T, color=c, lw=1, ls="--")
        ax.scatter(P[:, 0], P[:, 1], s=9, color=c)
    ax.plot(0, 0, "k+", ms=8)
    names = ", ".join(COLORS[m] for m in ovr) or "none"
    ax.set_title(f"hulls pairwise disjoint: {disjoint}    splits off: {names}\n{verdict}",
                 fontsize=10, loc="left")
    plt.show()


slider = lambda v, lo, hi, st: FloatSlider(value=v, min=lo, max=hi, step=st, continuous_update=False)
interact(show,
         a=slider(-2.5, -4.0, 0.0, 0.05),
         b=slider(1.3, -1.0, 2.0, 0.05),
         width=slider(0.3, 0.05, 1.0, 0.05),
         wobble=slider(0.15, 0.0, 0.4, 0.05),
         draw=Dropdown(options=["prefer one vs rest", "LP solution"], value="prefer one vs rest"));

interactive(children=(FloatSlider(value=-2.5, continuous_update=False, description='a', max=0.0, min=-4.0, ste…

With `width=0.3`, `wobble=0.15`, try

| $(a,b)$ | hulls pairwise disjoint | splits off | separable by |
|---|---|---|---|
| $(-2,\,0.3)$ | yes | all three | one vs rest, and also wedges |
| $(-2.5,\,1.1)$ | yes | none | wedges only |
| $(-2.5,\,1.3)$ | yes | none | **not separable** |

The `draw` menu picks which separating $W$ is shaded. *prefer one vs rest* shows the degenerate $W$ (two regions, $R_j=R_k$) whenever some class splits off; *LP solution* shows whatever $W$ the LP returns, which is typically a wedge $W$. The label in the title names the shape of the $W$ actually drawn. The two cases overlap: at $(-2,0.3)$ both kinds of $W$ exist.

Increasing `width` at $(-2.5,1.3)$ eventually makes the hulls overlap pairwise; decreasing $b$ at fixed width moves from not separable to wedges to one vs rest.